# Sistemas Expertos
## Inferencia lógica y técnicas de búsqueda


Ya vimos arquitectura, base de conocimiento, hechos, reglas, motor de inferencia y formas de representación. Hoy vamos a profundizar en **cómo razona un sistema** y cómo **busca soluciones**.

**Recorrido:** encadenamiento hacia adelante → encadenamiento hacia atrás → lógica AND/OR/NOT → BFS → DFS → resolución de problemas → actividad integradora.

## 0. Preparación

Los ejemplos usan situaciones de **construcción, mantenimiento e inspección de obras**.



In [ ]:
# Importamos NetworkX, una biblioteca que permite crear y trabajar con grafos.
import networkx as nx

# Importamos Matplotlib para poder realizar gráficos y visualizar los grafos.
import matplotlib.pyplot as plt

# Importamos deque, que nos permite trabajar con una cola.
# La vamos a utilizar posteriormente en el algoritmo BFS.
from collections import deque

# Mensaje para comprobar que el entorno se preparó correctamente.
print("Entorno preparado correctamente")

# 1. Puente con la clase anterior

Tenemos **hechos** y **reglas**. La inferencia es el proceso de obtener conclusiones a partir de ellos.

```text
HECHOS + REGLAS → INFERENCIA → CONCLUSIÓN
```

La pregunta de hoy es: **¿en qué orden utiliza el sistema esa información?**

# 2. Encadenamiento hacia adelante (Forward Chaining)

Partimos de los hechos conocidos y buscamos reglas cuyas condiciones se cumplan. Una conclusión nueva puede convertirse en un nuevo hecho y activar otra regla.

```text
Hechos iniciales → Regla → Nueva conclusión → otra regla → ...
```

In [ ]:
# Cada regla tiene un nombre, condiciones y una conclusión.
reglas_obra = [
    {"nombre":"R1", "condiciones":{"humedad_alta"}, "conclusion":"revisar_impermeabilizacion"},
    {"nombre":"R2", "condiciones":{"revisar_impermeabilizacion","lluvias_recientes"}, "conclusion":"inspeccionar_cubierta"},
    {"nombre":"R3", "condiciones":{"fisuras_importantes"}, "conclusion":"inspeccionar_estructura"},
    {"nombre":"R4", "condiciones":{"inspeccionar_estructura","edificio_antiguo"}, "conclusion":"solicitar_evaluacion_especializada"},
    {"nombre":"R5", "condiciones":{"temperatura_alta","humedad_alta"}, "conclusion":"revisar_condiciones_de_trabajo"},
    {"nombre":"R6", "condiciones":{"revisar_condiciones_de_trabajo","trabajadores_presentes"}, "conclusion":"reforzar_controles"}
]

hechos = {
    "humedad_alta", "lluvias_recientes", "fisuras_importantes",
    "edificio_antiguo", "temperatura_alta", "trabajadores_presentes"
}

print("Hechos iniciales:")
for h in sorted(hechos): print("-", h)

Este código crea la base de conocimiento del sistema experto y define los hechos con los que va a comenzar el razonamiento. Cada regla tiene un nombre, un conjunto de condiciones que deben cumplirse y una conclusión que se obtiene cuando esas condiciones se cumplen. Luego, hechos contiene la información inicial de la obra. Finalmente, el for recorre esos hechos y los muestra en pantalla.

In [ ]:
def encadenamiento_adelante(hechos_iniciales, reglas):
    hechos = set(hechos_iniciales)
    activadas = []
    cambio = True

    # Repetimos mientras aparezcan conclusiones nuevas.
    while cambio:
        cambio = False
        for regla in reglas:
            if regla["condiciones"].issubset(hechos) and regla["conclusion"] not in hechos:
                hechos.add(regla["conclusion"])
                activadas.append(regla["nombre"])
                print(f"{regla['nombre']} activada → {regla['conclusion']}")
                cambio = True
    return hechos, activadas

hechos_finales, activadas = encadenamiento_adelante(hechos, reglas_obra)
print("\nConclusiones finales:")
for h in sorted(hechos_finales): print("-", h)
print("Reglas activadas:", activadas)

Este código implementa el encadenamiento hacia adelante. Parte de los hechos iniciales y recorre las reglas buscando cuáles cumplen sus condiciones. Cuando una regla se cumple, agrega su conclusión como un nuevo hecho y registra la regla activada. El proceso se repite mientras aparezcan conclusiones nuevas. Al final devuelve todos los hechos obtenidos y las reglas que se activaron, y luego los muestra en pantalla.

## Actividad 1 — Ampliar la base de conocimiento

Agregá **3 reglas nuevas**. Al menos una debe depender de una conclusión obtenida por otra regla.

Ejemplos de conocimiento que podés incorporar: señalización, viento, EPP, iluminación, materiales almacenados al aire libre.

**Desafío:** conseguir un encadenamiento de al menos **3 pasos** y explicar oralmente qué hecho permitió activar cada regla.

In [ ]:
# Escribí aquí tus reglas nuevas y agregalas a reglas_obra.
# Ejemplo de estructura:
# nueva = {"nombre":"R7", "condiciones":{"..."}, "conclusion":"..."}
# reglas_obra.append(nueva)

# Luego ejecutá nuevamente el motor.


ACTIVIDAD 2— Encadenamiento hacia adelante
Creamos nuestras propias reglas

Elegí un tema que quieras para construir un pequeño sistema basado en reglas. Puede ser construcción, salud, deportes, videojuegos, clima, vehículos, educación, etc.

Consigna
Creá 3 reglas nuevas.
Cada regla debe tener:
un nombre;
condiciones;
una conclusión.
Al menos una regla debe utilizar como condición la conclusión obtenida por otra regla.
Incorporá las reglas a reglas_obra o creá una nueva base de conocimiento.
Ejecutá el encadenamiento hacia adelante.
Explicá qué reglas se activaron y en qué orden.
Desafío

Lográ un encadenamiento de 3 pasos:

Hecho inicial → Regla 1 → nueva conclusión → Regla 2 → nueva conclusión → Regla 3

Pregunta

¿Qué hecho o conclusión permitió que se activara cada nueva regla?

# 3. Encadenamiento hacia atrás (Backward Chaining)

Ahora empezamos por un **objetivo** y preguntamos qué reglas podrían demostrarlo. Después buscamos si se cumplen sus condiciones.

```text
OBJETIVO → regla que lo demuestra → condiciones → hechos
```

Ejemplo: para demostrar `solicitar_evaluacion_especializada`, necesitamos `inspeccionar_estructura` y `edificio_antiguo`. Para demostrar `inspeccionar_estructura`, necesitamos `fisuras_importantes`.

In [ ]:
def encadenamiento_atras(objetivo, hechos, reglas, camino=None):
    if camino is None:
        camino = []
    if objetivo in camino:
        return False

    if objetivo in hechos:
        print("  " * len(camino) + f"✓ {objetivo} es un hecho conocido")
        return True

    print("  " * len(camino) + f"→ Buscando cómo demostrar: {objetivo}")
    candidatas = [r for r in reglas if r["conclusion"] == objetivo]

    for regla in candidatas:
        print("  " * len(camino) + f"Probando {regla['nombre']}")
        if all(encadenamiento_atras(c, hechos, reglas, camino+[objetivo])
               for c in regla["condiciones"]):
            print("  " * len(camino) + f"✓ Objetivo demostrado: {objetivo}")
            return True

    print("  " * len(camino) + f"✗ No se pudo demostrar: {objetivo}")
    return False

objetivo = "solicitar_evaluacion_especializada"
print("Resultado:", encadenamiento_atras(objetivo, hechos, reglas_obra))

Este código implementa el encadenamiento hacia atrás. A diferencia del anterior, comienza con un objetivo y busca qué reglas pueden demostrarlo. Luego analiza las condiciones de esas reglas y comprueba si pueden demostrarse a partir de los hechos disponibles. Si todas las condiciones necesarias se cumplen, el objetivo queda demostrado; si no, el sistema informa que no pudo demostrarlo. Finalmente, se prueba el sistema con el objetivo solicitar_evaluacion_especializada.

## Actividad 3— Cambiar el objetivo

Probá con `inspeccionar_cubierta`, `inspeccionar_estructura`, `reforzar_controles` y un objetivo creado por vos.

Después eliminá un hecho necesario y volvé a ejecutar.

**Para discutir:** ¿qué diferencia observás entre partir de los hechos y partir del objetivo?

ACTIVIDAD 4 — Encadenamiento hacia atrás
Partimos de un objetivo

Ahora vamos a trabajar al revés.

En lugar de comenzar con los hechos y ver qué conclusiones podemos obtener, vamos a comenzar con una conclusión que queremos demostrar.

Consigna

Utilizá la base de conocimiento del ejercicio anterior.

Elegí una conclusión que quieras demostrar.
Utilizá encadenamiento_atras() para comprobar si puede demostrarse.
Observá qué reglas busca el sistema.
Identificá qué hechos necesita para llegar al objetivo.
Eliminá uno de los hechos necesarios.
Ejecutá nuevamente el sistema.
Responder
¿Cuál fue el objetivo?
¿Qué reglas utilizó para intentar demostrarlo?
¿Qué hechos necesitaba?
¿Qué ocurrió cuando eliminaste uno de ellos?
¿Qué diferencia observás entre comenzar por los hechos y comenzar por el objetivo?
Idea que tienen que descubrir

Encadenamiento hacia adelante:

Hechos → reglas → conclusión

Encadenamiento hacia atrás:

Objetivo → reglas → condiciones → hechos

# 4. Inferencia lógica: AND, OR y NOT

Las condiciones pueden combinarse.

- **AND:** deben cumplirse todas.
- **OR:** alcanza con una.
- **NOT:** se evalúa la ausencia/no cumplimiento de una condición.

Ejemplo: `humedad > 70 AND fisuras > 5` exige ambas condiciones.

In [ ]:
def evaluar_seguridad(humedad, fisuras, lluvia, senalizacion_correcta, usa_epp):
    recomendaciones = []

    if humedad > 70 and fisuras > 5:
        recomendaciones.append("Solicitar inspección urgente de la estructura")

    if humedad > 70 or lluvia:
        recomendaciones.append("Revisar posibles filtraciones")

    if not senalizacion_correcta:
        recomendaciones.append("Corregir la señalización del sector")

    if not usa_epp:
        recomendaciones.append("Verificar el uso de EPP")

    # Combinación: OR + AND
    if (humedad > 70 or lluvia) and fisuras > 3 and senalizacion_correcta:
        recomendaciones.append("Realizar inspección técnica prioritaria")

    return recomendaciones

resultado = evaluar_seguridad(78, 6, True, True, False)
for r in resultado: print("-", r)

Este código crea una función que evalúa las condiciones de seguridad de una obra y genera recomendaciones según los datos ingresados. Utiliza los operadores lógicos AND (and), OR (or) y NOT (not) para combinar o negar condiciones. Cada vez que una condición se cumple, se agrega una recomendación a la lista. Finalmente, la función devuelve todas las recomendaciones obtenidas y se prueba con un caso concreto: humedad de 78%, fisuras de 6 mm, lluvia, señalización correcta y sin uso de EPP.

## Actividad 5 — Diseñar reglas lógicas

Creá una función para evaluar una obra que tenga:

- 2 reglas con **AND**;
- 2 reglas con **OR**;
- 1 regla con **NOT**;
- 1 regla que combine **AND + OR**.

Probala con **4 escenarios diferentes**. La función debe devolver recomendaciones, no solamente `True/False`.

In [ ]:
# ACTIVIDAD 5
def evaluar_obra(temperatura, humedad, viento, fisuras, senalizacion_correcta, usa_epp):
    recomendaciones = []

    # Escribí tus reglas aquí.

    return recomendaciones

# Creá cuatro escenarios y compará las recomendaciones.


# 5. Resolución de problemas y espacio de búsqueda

Cuando hay varias alternativas, podemos representar el problema como un grafo.

Un problema de búsqueda tiene, como mínimo:

- **estado inicial**;
- **acciones posibles**;
- **estados alcanzables**;
- **objetivo**;
- una **estrategia de búsqueda**.

Ejemplo: un robot de inspección debe ir desde Entrada hasta un sector de Seguridad.

In [ ]:
G = nx.Graph()
conexiones = [
    ("Entrada","Recepción"), ("Entrada","Depósito"),
    ("Recepción","Oficina"), ("Recepción","Seguridad"),
    ("Depósito","Herramientas"), ("Depósito","Materiales"),
    ("Oficina","Administración"), ("Seguridad","Salida"),
    ("Herramientas","Taller"), ("Materiales","Taller")
]
G.add_edges_from(conexiones)

pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(11,6))
nx.draw(G, pos, with_labels=True, node_size=2200, node_color="lightblue", font_size=9)
plt.title("Mapa simplificado de una obra")
plt.show()

# 6. BFS — Búsqueda en amplitud

**Breadth-First Search** explora primero los nodos del nivel más cercano al inicio. Utiliza la lógica de una **cola (FIFO)**.

```text
Inicio
├── A
│   ├── C
│   └── D
└── B
    ├── E
    └── F
```

Primero explora A y B; luego C, D, E y F.

In [ ]:
def bfs(grafo, inicio, objetivo):
    cola = deque([(inicio, [inicio])])
    visitados = set()

    while cola:
        nodo, camino = cola.popleft()
        if nodo in visitados:
            continue
        visitados.add(nodo)
        print("Visitando:", nodo)

        if nodo == objetivo:
            return camino

        for vecino in grafo.neighbors(nodo):
            if vecino not in visitados:
                cola.append((vecino, camino + [vecino]))
    return None

camino_bfs = bfs(G, "Entrada", "Taller")
print("\nCamino BFS:", " → ".join(camino_bfs))

Este código implementa el algoritmo BFS (búsqueda en amplitud). Parte de un nodo inicial y utiliza una cola FIFO para recorrer el grafo por niveles. visitados evita recorrer un mismo nodo más de una vez, mientras que camino permite conservar la ruta seguida hasta cada nodo. El algoritmo continúa explorando hasta encontrar el objetivo; en este caso, busca un camino desde “Entrada” hasta “Taller” y finalmente muestra el recorrido encontrado.

# 7. DFS — Búsqueda en profundidad

**Depth-First Search** explora un camino en profundidad antes de retroceder. Utiliza una **pila (LIFO)**.

La diferencia importante no es solamente el camino final: también cambia el **orden en que se exploran los estados**.

DFS explora primero un camino lo más profundo posible y, cuando no puede continuar, retrocede para probar otra alternativa. Para hacerlo utiliza una pila LIFO, es decir, el último nodo que entra es el primero que se explora. Por eso, respecto de BFS, cambia principalmente el orden en que se recorren los estados.

In [ ]:
def dfs(grafo, inicio, objetivo):
    pila = [(inicio, [inicio])]
    visitados = set()

    while pila:
        nodo, camino = pila.pop()
        if nodo in visitados:
            continue
        visitados.add(nodo)
        print("Visitando:", nodo)

        if nodo == objetivo:
            return camino

        for vecino in grafo.neighbors(nodo):
            if vecino not in visitados:
                pila.append((vecino, camino + [vecino]))
    return None

camino_dfs = dfs(G, "Entrada", "Taller")
print("\nCamino DFS:", " → ".join(camino_dfs))

Este código implementa el algoritmo DFS (búsqueda en profundidad). Utiliza una pila LIFO para explorar un camino lo más profundo posible antes de retroceder. visitados evita repetir nodos y camino permite conservar la ruta recorrida. En este caso, busca un camino desde “Entrada” hasta “Taller” y muestra tanto el orden de exploración como el camino encontrado.

## Actividad 4 — Comparar BFS y DFS

Probá los dos algoritmos buscando:

- `Administración`
- `Salida`
- `Herramientas`
- `Materiales`
- `Taller`

Para cada objetivo registrá:
1. orden de visita;
2. camino encontrado;
3. cantidad de nodos visitados.

**No respondas “uno es mejor”.** Explicá cómo cambia la exploración según la estrategia.

# 8. Visualizar el camino encontrado

Visualizar el resultado ayuda a interpretar el algoritmo y no quedarse solamente con una lista de nombres.

In [ ]:
def mostrar_camino(grafo, camino, titulo):
    pos = nx.spring_layout(grafo, seed=42)
    plt.figure(figsize=(11,6))
    nx.draw(grafo, pos, with_labels=True, node_size=2200, node_color="lightgray", font_size=9)
    aristas = list(zip(camino, camino[1:]))
    nx.draw_networkx_edges(grafo, pos, edgelist=aristas, width=4)
    plt.title(titulo)
    plt.show()

mostrar_camino(G, camino_bfs, "Camino encontrado por BFS")
mostrar_camino(G, camino_dfs, "Camino encontrado por DFS")

# 9. Actividad integradora — Sistema inteligente de inspección

Una empresa quiere asistir la inspección de una obra. Hay datos sobre humedad, fisuras, temperatura, señalización y EPP. Además, un robot puede desplazarse entre sectores.

### Parte A — Conocimiento
Creá una base de al menos **8 reglas**, incluyendo reglas simples, AND, OR, NOT y reglas encadenadas.

### Parte B — Inferencia
Ejecutá encadenamiento hacia adelante y registrá qué reglas se activan.

### Parte C — Objetivo
Elegí una conclusión importante y utilizá encadenamiento hacia atrás para intentar demostrarla. Quitá después uno de los hechos necesarios y observá el cambio.

### Parte D — Búsqueda
Representá los sectores de la obra como grafo. Definí estado inicial, objetivo y conexiones. Ejecutá BFS y DFS.

### Parte E — Análisis
Respondé: ¿qué conclusiones obtuvo el sistema?, ¿qué reglas se activaron?, ¿qué cambió al partir del objetivo?, ¿qué recorridos hicieron BFS y DFS?, ¿por qué pueden diferir?, ¿qué información adicional necesitaría un sistema real?

# 10. Cierre del bloque de Sistemas Expertos

```text
CONOCIMIENTO
   ↓
HECHOS + REGLAS
   ↓
INFERENCIA
   ↓
CONCLUSIONES
````

Cuando el problema tiene alternativas:

```text
ESTADO INICIAL
   ↓
ESTADOS + ACCIONES
   ↓
BÚSQUEDA
   ↓
OBJETIVO / SOLUCIÓN
```

### Idea final
**Representar correctamente el conocimiento y el problema es tan importante como elegir el mecanismo que lo procesa.**